# Somo 05 - RAG ya Wakala


## Setup

Hii daftari inaonyesha muundo wa Agentic RAG (Retrieval-Augmented Generation) kwa kutumia Microsoft Agent Framework.

**Mambo yanayohitajika:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — anwani yako ya huduma ya Azure AI Search
- `AZURE_SEARCH_API_KEY` — ufunguo wako wa API wa Azure AI Search
- Usambazaji wa Azure OpenAI umewekwa kupitia vigezo vya mazingira
- Azure CLI imethibitishwa (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG ni nini?

RAG ya jadi hufuata mchakato uliowekwa: pata nyaraka, kisha tengeneza jibu. **Agentic RAG** inaenda zaidi kwa kumpa wakala uhuru wa kuamua **lini** na **jinsi** ya kupata taarifa.

Kwa Agentic RAG, wakala anaweza:
- **Kuamua** kama upatikanaji ni muhimu kabla ya kujibu swali
- **Kuchagua** chanzo cha data au zana ya kuuliza
- **Kutathmini** matokeo yaliyopatikana na kufanya upatikanaji wa ziada ikiwa jaribio la kwanza halitoshi
- **Kuunganisha** taarifa kutoka kwenye hatua mbalimbali za upatikanaji kuwa jibu linaloeleweka

Hii inafanya wakala kuwa na urahisi zaidi na usahihi ukilinganisha na mchakato wa kawaida wa kupata-basi-kuunda jibu.


## Kuunda Zana ya Utafutaji

Katika Agentic RAG, vyanzo vya data vya nje vinapakiwa kama **zana** ambazo wakala anaweza kuitumia anapohitaji. Hii inamwezesha wakala kutibu urejeshaji kama kitendo kingine anachoweza kuchukua, badala ya hatua ya lazima.

Hapo chini tunaelezea hifadhidata ya maarifa ya usafiri na kuiweka wazi kama zana ambayo wakala anaweza kuitumia kutafuta habari za marudio.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Kujenga Wakala wa RAG

Sasa tunaunda wakala ambaye ameagizwa **kushika taarifa kila wakati kabla ya kujibu**. Wakala hutumia zana ya `search_travel_knowledge` kuinua majibu yake kutoka kwenye hifadhidata ya maarifa badala ya kutegemea data za mafunzo yake mwenyewe.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Utafutaji wa Marudio — Mfumo wa Mtengenezaji-Mhakiki

Faida kuu ya Agentic RAG ni **utafutaji wa marudio**. Wakala anaweza kufanya mizunguko mingi ya utafutaji kuthibitisha, kuboresha, au kupanua kile alichokipata awali — sawa na utaratibu wa "mtengenezaji-mhakiki":

1. **Hatua ya mtengenezaji**: Wakala huchukua taarifa za awali na kuandika majibu ya msingi.
2. **Hatua ya mhakiki**: Wakala hufanya utafutaji za ziada kuthibitisha maelezo au kuziba mapengo.

Hapo chini, wakala huyo anaulizwa swali linalohitaji kulinganisha maeneo mengi, hali inayomsukuma kufanya utafutaji mara nyingi.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Muhtasari

Katika somo hili uli jifunza jinsi ya kujenga Mfumo wa **Agentic RAG** ukitumia Microsoft Agent Framework:

- **Agentic RAG** inaruhusu maajenti kuamua kwa uhuru lini waweke upya taarifa, na kufanya upokeaji kuwa wa mzunguko badala ya kuwa wa kudumu.
- **Zana kama vyanzo vya data**: Maktaba za maarifa za nje (kama Azure AI Search) zimefungwa kama zana ambazo maajenti wanaweza kuitumia.
- **Upokeaji wa mzunguko**: Muundo wa maker-checker unawawezesha maajenti kufanya mizunguko mingi ya upokeaji — kutafuta, kuthibitisha, na kusafisha — kabla ya kutoa jibu la mwisho.

Katika uzalishaji, ungebadilisha `TRAVEL_KNOWLEDGE_BASE` ndani ya kumbukumbu na faharasa halisi ya Azure AI Search kushughulikia upokeaji wa nyaraka kubwa za kusafiri.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
